# SecOps Sentinel — Keşifçi Veri Analizi ve Profilleme

Bu Jupyter Notebook, Kaggle Windows Event Log veri seti (`data/raw/eventlog.csv`) üzerinde kapsamlı profilleme analizi gerçekleştirir:
1. Makine ve Sunucu dağılımları
2. Olay Kaynağı (Source), Giriş Türü (EntryType) ve Kategori frekansları
3. Zaman kalıpları ve Mesai İçi / Mesai Dışı aktivite dağılımı
4. Şablonlama indirgeme oranı ve mükerrer log temizleme (deduplication)

In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = os.path.join('..', 'data', 'samples', 'sample_500.csv')
df = pd.read_csv(DATA_PATH)
print(f'Sample shape: {df.shape}')
df.head()

## 1. Makine Başına Olay Dağılımı

In [2]:
machine_counts = df['MachineName'].value_counts()
print(machine_counts)

## 2. Giriş Türü (EntryType) Analizi

In [3]:
entry_type_counts = df['EntryType'].value_counts()
print(entry_type_counts)

## 3. En Sık Görülen Olay Kaynakları (Top Sources)

In [4]:
top_sources = df['Source'].value_counts().head(10)
print(top_sources)

## 4. Mesaj Şablonlama Motoru (Template Extraction) Gösterimi

In [5]:
import re

def normalize_message(msg):
    if not msg or pd.isna(msg):
        return '<EMPTY>'
    msg = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP>', str(msg))
    msg = re.sub(r'\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b', '<GUID>', msg)
    msg = re.sub(r'\b0x[0-9a-fA-F]+\b', '<HEX>', msg)
    msg = re.sub(r'\b\d+\b', '<NUM>', msg)
    return msg.strip()

df['Template'] = df['Message'].apply(normalize_message)
unique_templates = df['Template'].nunique()
print(f'Original messages: {len(df)}, Unique templates: {unique_templates}, Ratio: {len(df) / unique_templates:.2f}:1')